In [ ]:
!gdown 1SH8B81L4DSeBNsA2vYJYhgVBzIUov2DW

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
daily_production = pd.read_parquet("daily_production.parquet")

In [ ]:
display(daily_production.tail())
daily_production.shape

In [ ]:
daily_production.info()

In [ ]:
daily_production.isna().sum()

In [ ]:
daily_production.describe()

In [ ]:
df_copy = daily_production.copy()

In [ ]:
initial_rows = len(df_copy)

# Create a temporary column containing the number of null values in each row
df_copy['null_count'] = df_copy.isnull().sum(axis=1)

# Sort by 'uwi', 'production_date', and 'null_count' (ascending so the one with the fewest nulls is first)
df_copy = df_copy.sort_values(by=['uwi', 'production_date', 'null_count'])

# Drop duplicates based on 'uwi' and 'production_date', keeping the first occurrence
df_copy = df_copy.drop_duplicates(subset=['uwi', 'production_date'], keep='first')

# Drop the temporary 'null_count' column
df_copy = df_copy.drop(columns=['null_count'])

final_rows = len(df_copy)
print(f"Initial rows: {initial_rows}")
print(f"Rows after deduplication: {final_rows}")
print(f"Removed {initial_rows - final_rows} duplicate rows.")

In [ ]:
# Convert decimal128 / string columns to float
for col in ['surface_latitude', 'surface_longitude', 'pcp_pump_rpm']:
    df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')

# Parse date columns to datetime
for col in ['on_production_date', 'production_date']:
    df_copy[col] = pd.to_datetime(df_copy[col], errors='coerce')

# Display data types of the converted columns and confirm changes
display(df_copy[['surface_latitude', 'surface_longitude', 'pcp_pump_rpm', 'on_production_date', 'production_date']].dtypes)
display(df_copy[['surface_latitude', 'surface_longitude', 'pcp_pump_rpm', 'on_production_date', 'production_date']].head())

In [ ]:
#Replace sentinels 999999 and 999 with NaN
# We apply this to columns that frequently use these placeholders (pressures, pump efficiency, polymer etc.)
placeholders = [999999, 999999.0, 999, 999.0]
df_copy = df_copy.replace(placeholders, np.nan)

#Handle negative volumes, hours, and pressures
numeric_cols = df_copy.select_dtypes(include=[np.number]).columns

# Specific columns to correct for negative values (making them NaN)
neg_correction_cols = [
    'hours_down', 'hours_on_prod', 'lost_oil_m3',
    'tubing_pressure_kpa', 'casing_pressure_kpa', 'injection_pressure_kpa',
    'oil_produced_m3', 'gas_produced_e3m3', 'injection_volume_m3',
    'injected_water_m3', 'injected_polymer_m3', 'water_produced_m3',
    'hydraulic_pressure_psi'
]

for col in neg_correction_cols:
    if col in df_copy.columns:
        df_copy.loc[df_copy[col] < 0, col] = np.nan

#Handle choke_open_pct > 1 or 100
# If values are above 100, we'll set them to NaN. If they are in the [1, 100] scale, we standardise them to [0, 1]
if 'choke_open_pct' in df_copy.columns:
    # Set values clearly above 100 to NaN
    df_copy.loc[df_copy['choke_open_pct'] > 100, 'choke_open_pct'] = np.nan
    # Convert percentage (0-100) to decimal (0-1)
    df_copy.loc[df_copy['choke_open_pct'] > 1.0, 'choke_open_pct'] = df_copy['choke_open_pct'] / 100.0

# Handle pump parameters (RPM, amps, torque, throughput) above physical limits using 99.9th percentile caps
pump_cols = [
    'pcp_pump_rpm', 'pcp_pump_electric_current_a',
    'pcp_pump_rod_torque_ftlb', 'pcp_pump_volume_throughput_m3d'
]

for col in pump_cols:
    if col in df_copy.columns:
        # Determine 99.9th percentile threshold for physical limits
        threshold = df_copy[col].quantile(0.999)
        print(f"Capping {col} at 99.9th percentile: {threshold}")
        df_copy.loc[df_copy[col] > threshold, col] = np.nan

# Check the results
display(df_copy[pump_cols + ['choke_open_pct']].describe())

In [ ]:
# Standardise categoricals across object columns (e.g. primary_fluid_type, well_status, well_daily_status)
categorical_cols = ['primary_fluid_type', 'well_status', 'well_daily_status']
for col in categorical_cols:
    if col in df_copy.columns:
        df_copy[col] = df_copy[col].replace({'Oil': 'Heavy Oil', 'H2O Inj.': 'Water Injection'})

# Check well_status against well_daily_status, then drop well_status
print("Comparing well_status and well_daily_status unique pairs before dropping:")
if 'well_status' in df_copy.columns and 'well_daily_status' in df_copy.columns:
    print(df_copy.groupby(['well_status', 'well_daily_status'], dropna=False).size().head(20))
    df_copy = df_copy.drop(columns=['well_status'])

# Fill null statuses in well_daily_status using flow_direction, production hours/volumes, or drop them
print(f"\nNulls in well_daily_status before filling: {df_copy['well_daily_status'].isna().sum()}")

# Helper function to infer status based on other fields
def infer_status(row):
    if pd.notna(row['well_daily_status']):
        return row['well_daily_status']

    # Use flow_direction to infer
    if pd.notna(row['flow_direction']):
        if row['flow_direction'] == 'Injection':
            return 'Water Injection' # Safe default mapping for injection direction
        elif row['flow_direction'] == 'Production':
            return 'Heavy Oil' # Safe default mapping for production direction

    # Use hours and volumes
    if row['hours_on_prod'] > 0:
        if row['oil_produced_m3'] > 0 or row['gas_produced_e3m3'] > 0 or row['water_produced_m3'] > 0:
            return 'Heavy Oil'
        elif row['injection_volume_m3'] > 0 or row['injected_water_m3'] > 0:
            return 'Water Injection'

    return np.nan if 'np' in globals() else None

# Apply filling logic
import numpy as np
df_copy['well_daily_status'] = df_copy.apply(infer_status, axis=1)

# Drop remaining rows where well_daily_status is still null
df_copy = df_copy.dropna(subset=['well_daily_status'])

print(f"Nulls in well_daily_status after filling and dropping: {df_copy['well_daily_status'].isna().sum()}")
print(f"Remaining rows in df_copy: {len(df_copy)}")

In [ ]:
if 'pump_efficiency_pct' in df_copy.columns:
    print("Summary of pump_efficiency_pct before correction:")
    print(df_copy['pump_efficiency_pct'].describe())

    # Where pump_efficiency_pct > 1, divide by 100
    df_copy.loc[df_copy['pump_efficiency_pct'] > 1, 'pump_efficiency_pct'] /= 100.0

    # Set values above 1.2 to NaN (as requested: 'set values above that to NaN')
    df_copy.loc[df_copy['pump_efficiency_pct'] > 1.2, 'pump_efficiency_pct'] = np.nan

    # Clip to [0, 1.2]
    df_copy['pump_efficiency_pct'] = df_copy['pump_efficiency_pct'].clip(lower=0, upper=1.2)

    print("\nSummary of pump_efficiency_pct after correction:")
    print(df_copy['pump_efficiency_pct'].describe())
else:
    print("Column 'pump_efficiency_pct' not found in DataFrame.")

In [ ]:
#Check throughput ≈ displacement * RPM / 100 and use it to flag/fill bad pump readings
print("Columns in DataFrame:", [col for col in ['throughput', 'displacement', 'rpm', 'pump_efficiency_pct'] if col in df_copy.columns])

# Check if necessary columns exist before performing the validation
needed_cols = ['displacement', 'rpm']
if all(col in df_copy.columns for col in needed_cols):
    # Calculate theoretical displacement throughput
    # Note: displacement is typically volume per revolution, RPM is revolutions per minute
    # We calculate the expected baseline based on the formula provided:
    df_copy['expected_throughput_est'] = (df_copy['displacement'] * df_copy['rpm']) / 100.0

    # We can flag bad readings where actual throughput or pump efficiency significantly deviates
    # or is physically impossible given the displacement and RPM
    if 'throughput' in df_copy.columns:
        # Flag rows where reported throughput exceeds theoretical max capacity by a threshold (e.g., > 1.2 * expected)
        df_copy['bad_pump_reading_flag'] = (df_copy['throughput'] > 1.2 * df_copy['expected_throughput_est'])
        print(f"Number of flagged bad pump readings: {df_copy['bad_pump_reading_flag'].sum()}")
    else:
        print("\n'throughput' column not found, but 'expected_throughput_est' has been computed.")
else:
    print("\nRequired columns for pump calculation (displacement, rpm) are not present in df_copy.")

In [ ]:
#Fill volume columns (oil, water, gas) and lost_oil_m3 with 0 where status is Suspended or Abandoned.
# Leave them as NaN where status is Producing and the value is missing.

volume_cols = ['oil_produced_m3', 'water_produced_m3', 'gas_produced_e3m3', 'lost_oil_m3']
existing_vol_cols = [col for col in volume_cols if col in df_copy.columns]

if 'well_daily_status' in df_copy.columns and existing_vol_cols:
    print("Missing values before filling:")
    print(df_copy[existing_vol_cols].isna().sum())

    # Identify rows where status is 'Suspended' or 'Abandoned'
    suspended_or_abandoned = df_copy['well_daily_status'].isin(['Suspended', 'Abandoned'])

    # Fill with 0 for these specific rows across all existing volume columns
    for col in existing_vol_cols:
        df_copy.loc[suspended_or_abandoned & df_copy[col].isna(), col] = 0.0

    print("\nMissing values after filling (Producing rows remain NaN if missing):")
    print(df_copy[existing_vol_cols].isna().sum())
else:
    print("Required columns for volume filling are missing.")

In [ ]:
# Export the cleaned DataFrame to a new parquet file
saved_path = "/content/cleaned_daily_production.parquet"
df_copy.to_parquet(saved_path, index=False)
print(f"Cleaned dataset successfully saved to: {saved_path}")